Hello, my name is Aaron Kann.  In this data analysis, I seek to determine the point in the season in which a player's year to date averages become a better predictor of future outputs than a players previous year averages.

I expect that, in the first couple of weeks into a Premier League Season, a player's previous averages are a stronger predictive metric because a player's to date averages are suggested to a much greater degree of variance.  However, I hypothesize that around matchweek 6, a player's to date averages become a stronger metric because the recency advantage outweighs the larger variance.  

In addition, I expect that for metrics with a lower total count (goals, assists, yellow cards, etc.), last years output may be a more relevant factor for longer than metrics with a greater total count (total passes, shots attempted, clearances etc), because the high variance is magnified in these cases. 

Let's see if the data supports my hypotheses!

In [1]:
import csv, os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Read the data from the csv file
if not 'Premier_League_Team_Stats.csv' in os.listdir():
    os.chdir('..')
import fbref_scrape as fbref
from constants import *


In [2]:
def add_name_to_df(df, file):
    name = file.split('Matchlogs')[0]
    # get rid of the last underscore, and change all other underscores to spaces
    name = name[:-1].replace('_', ' ')
    # add the name all rows in the dataframe
    df['Name'] = name
    return df

In [3]:
prev_yr_df = pd.read_csv('Premier_League_Player_Stats.csv')

data_folder = 'Premier_League_Player_Matchlogs'
data_files = os.listdir(data_folder)
dfs = []
dfs_names = []

for file in data_files:
    if file.endswith('.csv'):
        file_path = os.path.join(data_folder, file)
        # read the csv file and process the data
        df = pd.read_csv(file_path)
        df = add_name_to_df(df, file)
        dfs.append(df)
        dfs_names.append(file)

dfs[0].head()

,game_started,game_count,dayofweek,comp,round,venue,result,team,opponent,position,...,passes_completed,passes,passes_pct,progressive_passes,carries,progressive_carries,take_ons,take_ons_won,match_report,Name
0,Y,1,Mon,Premier League,Matchweek 1,Home,W 1-0,Manchester Utd,Wolves,RB,...,47,51,92.2,3,42,1,4,3,Match Report,Aaron Wan Bissaka
1,Y,2,Sat,Premier League,Matchweek 2,Away,L 0-2,Manchester Utd,Tottenham,RB,...,38,44,86.4,1,29,1,2,1,Match Report,Aaron Wan Bissaka
2,Y,3,Sat,Premier League,Matchweek 3,Home,W 3-2,Manchester Utd,Nott'ham Forest,RB,...,58,69,84.1,9,55,7,2,1,Match Report,Aaron Wan Bissaka
3,Y,4,Sun,Premier League,Matchweek 4,Away,L 1-3,Manchester Utd,Arsenal,RB,...,44,54,81.5,2,30,2,0,0,Match Report,Aaron Wan Bissaka
4,N,5,Sat,Premier League,Matchweek 5,Home,L 1-3,Manchester Utd,Brighton,CB,...,10,11,90.9,1,5,0,0,0,Match Report,Aaron Wan Bissaka


In [4]:
def update_avgs(season_avgs, row, index):
    #iterate through the stats in season_avgs and update the averages
    for stat in season_avgs.keys():
        if stat in row.keys():
            if stat == "team":
                continue
            season_avgs[stat] = (season_avgs[stat]*index + row[stat])/(index+1)
    return season_avgs

# Get a player name from the matchlogs file name
def get_player_name(filename):
    return filename.split("_Matchlogs")[0].replace('_', ' ')

def get_player_prev_yr_statline(prev_yr_df, player_name):
    return prev_yr_df[prev_yr_df['name'] == player_name]

In [5]:
stats_wanted = fbref.get_stats_wanted('fbref')
stats_wanted.remove('name')
stats_wanted.remove('team')
stats_wanted.append('game_count')
df_as_list = list()

for i in range(len(dfs)): #iterates through each player's data
    
    # create a new row, with the season totals as of matchweek 0 (ie all zero)
    season_avgs = pd.Series(data=[0]*len(stats_wanted), index=stats_wanted)
    # print(season_avgs, last_five_avgs)


    for index, row in dfs[i].iterrows(): #iterates through each game of each players data
        # get rid of the buggy goalie statlines, which have no stats TODO: fix this
        if "take_ons_won" not in row.keys():
            continue

        dfrow = pd.concat([row, season_avgs.add_suffix("_szn")], axis=0)
        season_avgs = update_avgs(season_avgs, row, index)

        
        #add last season averages to the list
        dfrow['player_name'] = get_player_name(dfs_names[i])
        prev_yr_statline = get_player_prev_yr_statline(prev_yr_df, dfrow['player_name'])
        
        #iterate through the stats in prev_yr_statline and add them to the row
        if prev_yr_statline.empty:
            continue
        else:
            for stat in prev_yr_statline.keys():
                if prev_yr_statline["games"].iloc[0] != 0 and type(prev_yr_statline[stat].iloc[0]) ==  np.int64:
                    # print(stat)
                    dfrow[stat + "_prev_yr"] = prev_yr_statline[stat].iloc[0] / prev_yr_statline['games'].iloc[0]
                else:
                    dfrow[stat + "_prev_yr"] = prev_yr_statline[stat].iloc[0]
            
        #add the row to the list        
        df_as_list.append(dfrow)

    #end game loop
#end player loop

df = pd.DataFrame(df_as_list)
df.head()

C:\Users\Aaron\AppData\Local\Temp\ipykernel_21788\3171090442.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '77.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  season_avgs[stat] = (season_avgs[stat]*index + row[stat])/(index+1)
C:\Users\Aaron\AppData\Local\Temp\ipykernel_21788\3171090442.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '89.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  season_avgs[stat] = (season_avgs[stat]*index + row[stat])/(index+1)
C:\Users\Aaron\AppData\Local\Temp\ipykernel_21788\3171090442.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '46.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype 

,game_started,game_count,dayofweek,comp,round,venue,result,team,opponent,position,...,crosses_prev_yr,shots_prev_yr,shots_on_target_prev_yr,games,gk_saves,clearances,assisted_shots,fouled,crosses,fouls
0,Y,1,Mon,Premier League,Matchweek 1,Home,W 1-0,Manchester Utd,Wolves,RB,...,0.0,0.526316,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Y,2,Sat,Premier League,Matchweek 2,Away,L 0-2,Manchester Utd,Tottenham,RB,...,0.0,0.526316,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Y,3,Sat,Premier League,Matchweek 3,Home,W 3-2,Manchester Utd,Nott'ham Forest,RB,...,0.0,0.526316,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Y,4,Sun,Premier League,Matchweek 4,Away,L 1-3,Manchester Utd,Arsenal,RB,...,0.0,0.526316,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,N,5,Sat,Premier League,Matchweek 5,Home,L 1-3,Manchester Utd,Brighton,CB,...,0.0,0.526316,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
df.to_csv('prev_yr_output.csv', index=False)

In [7]:
# Iterate though each of the matchweeks, performing a linear regression on the data
for i in range(38):
    if i in [0, 1]:
        continue
    # Get the data for the current matchweek
    matchweek_data = df[df['game_count'] == i]
    # Perform a linear regression on the data
    # Get the x and y values
    Y = np.array(matchweek_data['passes'])
    X = np.array(matchweek_data[['passes_prev_yr', 'passes_szn']])
    X = sm.add_constant(X)

    model = sm.OLS(Y, X).fit()
    
    
    if (i in [2, 4, 6, 11, 19, 28]):
        print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.392
Model:                            OLS   Adj. R-squared:                  0.387
Method:                 Least Squares   F-statistic:                     74.24
Date:                Wed, 21 Jan 2026   Prob (F-statistic):           1.33e-25
Time:                        08:29:02   Log-Likelihood:                -1018.7
No. Observations:                 233   AIC:                             2043.
Df Residuals:                     230   BIC:                             2054.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         17.3394      2.057      8.430      0.0